# 03 · Gold: agregação para análise

Este notebook constrói a camada **gold**. Ele junta empréstimos e clientes da silver e cria a tabela `growdev.gold.resumo_clientes`, **pronta para análise de negócio**. A bronze guarda o dado bruto, a silver o dado limpo e a gold o dado **resumido**, que é a camada consultada por dashboards e relatórios.

**Parâmetros**
- `catalogo`: catálogo do projeto.
- `simular_falha`: `sim` faz o notebook falhar de propósito. Usado na aula 19 (Repair run).

**Task value publicado:** `taxa_inadimplencia_pct`, usado no If/else do lab da aula 17.

In [ ]:
dbutils.widgets.text("catalogo", "growdev", "Catálogo")
dbutils.widgets.dropdown("simular_falha", "nao", ["nao", "sim"], "Simular falha?")

CATALOGO = dbutils.widgets.get("catalogo")
SIMULAR_FALHA = dbutils.widgets.get("simular_falha")

In [ ]:
# Em produção isso não existe! Serve só para treinarmos o diagnóstico e o Repair run.
if SIMULAR_FALHA == "sim":
    raise Exception(
        "Falha simulada: a tabela de referência de clientes não está disponível. "
        "Corrija o parâmetro simular_falha para 'nao' e use o Repair run."
    )

In [ ]:
from pyspark.sql import functions as F

SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"

df_loans = spark.table(f"{CATALOGO}.{SCHEMA_SILVER}.loans")
df_customers = spark.table(f"{CATALOGO}.{SCHEMA_SILVER}.customers")

print(f"loans: {df_loans.count()} linhas")
print(f"customers: {df_customers.count()} linhas")

In [ ]:
# 1. Junta empréstimos com clientes pelo customer_id
# 2. Agrupa por cliente e calcula as métricas
df_gold = (
    df_loans.join(df_customers, "customer_id")
    .groupBy("customer_id", "full_name", "state", "customer_segment")
    .agg(
        F.count("loan_id").alias("qtd_emprestimos"),
        F.sum("loan_amount").alias("valor_total_emprestado"),
        F.sum(F.when(F.col("is_default"), 1).otherwise(0)).alias("qtd_inadimplencias"),
    )
)

tabela_gold = f"{CATALOGO}.{SCHEMA_GOLD}.resumo_clientes"

# A gold é sempre recalculada por completo: overwrite substitui o resultado anterior
df_gold.write.mode("overwrite").saveAsTable(tabela_gold)

print(f"OK: {tabela_gold} criada com {df_gold.count()} clientes")

In [ ]:
resumo = spark.table(tabela_gold).agg(
    F.sum("qtd_emprestimos").alias("emprestimos"),
    F.sum("qtd_inadimplencias").alias("inadimplencias"),
).first()

taxa_inadimplencia_pct = round(100 * resumo["inadimplencias"] / resumo["emprestimos"], 2)
print(f"Taxa de inadimplência da carteira: {taxa_inadimplencia_pct}%")

# Publica o valor para as próximas tarefas do job (ex.: um If/else).
# Quando o notebook é chamado por outro (dbutils.notebook.run), não existe tarefa de job
# para receber o recado, então seguimos sem publicar.
try:
    dbutils.jobs.taskValues.set(key="taxa_inadimplencia_pct", value=taxa_inadimplencia_pct)
except Exception:
    print("Fora de uma tarefa de job: task value não publicado.")

In [ ]:
df_resumo = spark.table(tabela_gold)

# Clientes com maior valor total emprestado
display(df_resumo.orderBy(F.desc("valor_total_emprestado")).limit(10))

# Inadimplência média por segmento de cliente
display(
    df_resumo.groupBy("customer_segment")
    .agg(
        F.count("*").alias("clientes"),
        F.avg("qtd_inadimplencias").alias("inadimplencias_media"),
    )
    .orderBy(F.desc("clientes"))
)